# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**The rule**: a visible page (impressions_90d >= 500) gets flagged into one of three archetypes. A CTR gap page has clicks below what its position tier normally earns. A tracking gap page shows exactly zero CTR at real volume, more likely an instrumentation problem than a writing problem. A decay risk page sits on page one but hasn't been touched in 180+ days. Within the CTR gap archetype, pages rank by gap size dampened by audience size, not gap times audience raw, so one huge page with an ordinary gap can no longer outrank several genuinely large gaps on smaller pages. A page hitting two archetypes at once gets flagged as combined priority rather than counted twice.

In [2]:
import os, subprocess
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/solasobambo-prog/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")

Loaded 30,000 rows


In [3]:
# Same tier_avg_ctr / ctr_gap logic as w04_baseline_score.ipynb, Section 2.
# Tier average is computed only on the trustworthy slice (impressions_90d >= 100),
# then mapped onto every row, including ones below that floor, so they still get
# a fair comparison point.
trustworthy = df[df["impressions_90d"] >= 100]
tier_avg_ctr = trustworthy.groupby("position_tier")["ctr"].mean()

df["tier_avg_ctr"] = df["position_tier"].map(tier_avg_ctr)
df["ctr_gap"] = df["ctr"] - df["tier_avg_ctr"]

print("Tier average CTR:")
print(tier_avg_ctr.round(3))
print()
print(f"Rows with tier_avg_ctr still missing (position_tier unseen in trustworthy slice): {df['tier_avg_ctr'].isna().sum()}")

Tier average CTR:
position_tier
deep        0.055
page_1      0.355
page_3_5    0.142
striking    0.256
top_3       0.334
Name: ctr, dtype: float64

Rows with tier_avg_ctr still missing (position_tier unseen in trustworthy slice): 0


In [11]:
import numpy as np

TRUST_FLOOR = 500

# --- Archetype flags, built on columns you already have from w04/w05 ---
visible = df["impressions_90d"] >= TRUST_FLOOR

is_ctr_gap = visible & (df["ctr"] > 0) & (df["ctr"] < df["tier_avg_ctr"])
is_tracking_gap = visible & (df["ctr"] == 0)
is_decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) \
                & (df["content_age_days"] >= 180)

df["is_combined_priority"] = is_ctr_gap & is_decay_risk

# --- Reason code + action label, priority order matters: check combined first ---
conditions = [
    is_ctr_gap & is_decay_risk,
    is_ctr_gap,
    is_tracking_gap,
    is_decay_risk,
]
reason_codes = [
    "ctr_below_tier_visible_and_decay_risk",
    "ctr_below_tier_visible",
    "possible_tracking_gap",
    "page_one_decay_risk",
]
actions = [
    "priority_review_ctr_and_refresh",
    "review_for_ctr_gap",
    "verify_tracking_setup",
    "schedule_content_refresh",
]

df["reason_code"] = np.select(conditions, reason_codes, default="none")
df["action"] = np.select(conditions, actions, default="none")

# --- Ranking key: dampened, fixes the volume-inflation weak pick from ML-07 ---
df["ctr_gap_abs"] = (df["ctr"] - df["tier_avg_ctr"]).abs()
df["rank_score"] = df["ctr_gap_abs"] * np.log1p(df["impressions_90d"])

# --- Estimated impact: shown as a value column, never the sort key itself ---
df["estimated_impact"] = df["impressions_90d"] * df["ctr_gap_abs"]

# --- Build the queue: only flagged rows, ranked within archetype ---
queue = df[df["reason_code"] != "none"].copy()
queue = queue.sort_values(["reason_code", "rank_score"], ascending=[True, False])

review_cols = ["content_id", "archetype" if "archetype" in queue else "reason_code",
               "action", "content_type", "position_tier", "impressions_90d",
               "ctr", "ctr_gap_abs", "rank_score", "estimated_impact",
               "content_age_days"]

print("Archetype counts:")
print(queue["reason_code"].value_counts())
print()
print("Top 10 overall by rank_score, CTR gap archetype only:")
print(queue[queue["reason_code"] == "ctr_below_tier_visible"]
      .head(10)[review_cols].to_string(index=False))

# --- Content-type breakout, so keyword article doesn't crowd out everything else ---
print()
print("Top 5 per content_type, so smaller types stay visible:")
print(queue.groupby("content_type")
      .apply(lambda g: g.head(5))[review_cols])

Archetype counts:
reason_code
ctr_below_tier_visible                   5862
page_one_decay_risk                      4200
ctr_below_tier_visible_and_decay_risk    2616
possible_tracking_gap                    2538
Name: count, dtype: int64

Top 10 overall by rank_score, CTR gap archetype only:
          content_id            reason_code             action    content_type position_tier  impressions_90d  ctr  ctr_gap_abs  rank_score  estimated_impact  content_age_days
content_453722754fea ctr_below_tier_visible review_for_ctr_gap keyword article        page_1           140079 0.01     0.344760    4.085391      48293.586064                97
content_39881853ef0c ctr_below_tier_visible review_for_ctr_gap keyword article        page_1           112434 0.01     0.344760    4.009600      38762.705727                97
content_c84a0ab98e90 ctr_below_tier_visible review_for_ctr_gap keyword article        page_1           223271 0.03     0.324760    3.999787      72509.410303                95
c

/tmp/ipykernel_2317/314354305.py:65: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(5))[review_cols])


In [5]:
ctr_queue = queue[queue["reason_code"] == "ctr_below_tier_visible"].sort_values("rank_score", ascending=False)
top_per_type = ctr_queue.groupby("content_type", group_keys=False).head(5)
print(top_per_type[review_cols])

                 content_id             reason_code              action  \
27178  content_453722754fea  ctr_below_tier_visible  review_for_ctr_gap   
482    content_39881853ef0c  ctr_below_tier_visible  review_for_ctr_gap   
6903   content_c84a0ab98e90  ctr_below_tier_visible  review_for_ctr_gap   
3394   content_36ff89c8214e  ctr_below_tier_visible  review_for_ctr_gap   
9193   content_c1fe78bc4e37  ctr_below_tier_visible  review_for_ctr_gap   
17062  content_ea35d7b74f0d  ctr_below_tier_visible  review_for_ctr_gap   
9628   content_acaab4530f70  ctr_below_tier_visible  review_for_ctr_gap   
9414   content_5f27fe74502d  ctr_below_tier_visible  review_for_ctr_gap   
34     content_55f75c034970  ctr_below_tier_visible  review_for_ctr_gap   
26669  content_503edccc7cfa  ctr_below_tier_visible  review_for_ctr_gap   
28975  content_57aca9f1a5fe  ctr_below_tier_visible  review_for_ctr_gap   
7712   content_67d3b06ae42b  ctr_below_tier_visible  review_for_ctr_gap   
28316  content_a73de2878d

In [12]:
import numpy as np

TRUST_FLOOR = 500
DECAY_IMPRESSIONS_FLOOR = 100

visible = df["impressions_90d"] >= TRUST_FLOOR

is_ctr_gap = visible & (df["ctr"] > 0) & (df["ctr"] < df["tier_avg_ctr"])
is_tracking_gap = visible & (df["ctr"] == 0)
is_decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) \
                & (df["content_age_days"] >= 180) \
                & (df["impressions_90d"] >= DECAY_IMPRESSIONS_FLOOR)

conditions = [
    is_ctr_gap & is_decay_risk,
    is_ctr_gap,
    is_tracking_gap,
    is_decay_risk,
]
reason_codes = [
    "ctr_below_tier_visible_and_decay_risk",
    "ctr_below_tier_visible",
    "possible_tracking_gap",
    "page_one_decay_risk",
]
actions = [
    "priority_review_ctr_and_refresh",
    "review_for_ctr_gap",
    "verify_tracking_setup",
    "schedule_content_refresh",
]

df["reason_code"] = np.select(conditions, reason_codes, default="none")
df["action"] = np.select(conditions, actions, default="none")
df["ctr_gap_abs"] = (df["ctr"] - df["tier_avg_ctr"]).abs()
df["rank_score"] = df["ctr_gap_abs"] * np.log1p(df["impressions_90d"])
df["estimated_impact"] = df["impressions_90d"] * df["ctr_gap_abs"]

queue = df[df["reason_code"] != "none"].copy()
queue = queue.sort_values(["reason_code", "rank_score"], ascending=[True, False])

print("Archetype counts, with the new decay floor applied:")
print(queue["reason_code"].value_counts())
print()
print(f"Total flagged: {len(queue):,} of {len(df):,} ({len(queue)/len(df):.1%})")

Archetype counts, with the new decay floor applied:
reason_code
ctr_below_tier_visible                   5862
ctr_below_tier_visible_and_decay_risk    2616
possible_tracking_gap                    2538
page_one_decay_risk                      2047
Name: count, dtype: int64

Total flagged: 13,063 of 30,000 (43.5%)


**Confirmed on the data**: the three archetypes preserve the original rule's population exactly (5,862 + 2,616 + 2,538 = 11,016, matching w04's flagged count), splitting it into finer reason codes rather than changing who gets flagged. Adding the decay archetype grows total flagged pages to 15,216 (50.7%), nearly all page_1 or striking tier. The per content type breakout shows keyword article's opportunities are an order of magnitude larger by volume than comparison or feedly article's, so a combined ranking alone would leave the latter two invisible.

Decay risk additionally requires impressions_90d >= 100, the same trust floor used elsewhere in this rule for tier average CTR. That removed 2,153 low traffic rows from the standalone decay archetype (4,200 down to 2,047), while the combined CTR gap and decay bucket held steady at 2,616, confirming those pages had already cleared the higher 500 impression floor through the CTR side. Total flagged is now 13,063 of 30,000 (43.5%).

## 2. Intended use and limits

**Who uses this:** content editors and SEO strategists work the `review_for_ctr_gap` and
combined priority rows to check titles, meta descriptions, and snippets. Engineering or
analytics owns `possible_tracking_gap` rows, since a real zero CTR at volume needs a
tagging or parameter check before anyone touches the content itself. Content strategy or
planning owns `schedule_content_refresh` rows for capacity planning, not immediate action.
It is not intended for automated publishing, for grading individual writers' output, or
for any client facing promise about what a review will recover.

**Where it stops being valid:**

- **Sample, not full population.** This runs on the 30,000 row anonymized starter slice
  across 32 clients, not the roughly 520,000 page warehouse release. Numbers here describe
  this sample, not FlyRank's full inventory.
- **A rule, deliberately, not a model.** Week 5's honestly validated Random Forest scored a
  negative R squared under a grouped split, so this playbook stays rule based on purpose,
  archetype membership from fixed thresholds, no probability or confidence score attached.
- **Tier average CTR is a snapshot, not a live number,** computed once from this pull. If
  search behavior shifts, the tiers stop reflecting reality until recomputed, a limit today
  and a trigger in Section 4.
- **Decay risk now carries the same 100 impression floor used for tier average CTR
  elsewhere in this rule,** removing 2,153 of the original 4,200 standalone decay picks.
  The remaining 2,047 are pages a reviewer can trust have real traffic behind them, not
  just old rank.
- **The CTR gap archetype under-covers `top_3` and `deep` tiers.** Together they're under
  3% of the archetype (2.2% and 0.6%), while `striking`, `page_3_5`, and `page_1` are each
  roughly a third. Not a page_1 concentration problem, a `top_3`/`deep`
  under-representation problem.
- **Page grain, not query grain.** The rule flags a page, not which queries drive the
  shortfall, an editor still needs Search Console to decide what to change.
- **No causal or outcome claim.** An archetype says where to look, not what's wrong or that
  a fix will work, consistent with the `review_for_ctr_gap` rename from ML-09.

In [6]:
# Supports the "sample, not full population" and "tier concentration" claims above
print(f"Unique clients in this sample: {df['client_id'].nunique()}")
print(f"Date range represented: trailing 90-day window, single snapshot")
print()
print("Position tier share of the CTR-gap archetype:")
print(queue[queue["reason_code"] == "ctr_below_tier_visible"]["position_tier"].value_counts(normalize=True).round(3))
print()
print("Decay-risk rows with impressions_90d < 100 (no floor currently applied):")
low_traffic_decay = queue[(queue["reason_code"].str.contains("decay_risk")) & (queue["impressions_90d"] < 100)]
print(f"{len(low_traffic_decay):,} of {(queue['reason_code'].str.contains('decay_risk')).sum():,}")

Unique clients in this sample: 32
Date range represented: trailing 90-day window, single snapshot

Position tier share of the CTR-gap archetype:
position_tier
striking    0.376
page_3_5    0.305
page_1      0.291
top_3       0.022
deep        0.006
Name: proportion, dtype: float64

Decay-risk rows with impressions_90d < 100 (no floor currently applied):
2,153 of 6,816


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [18]:
import numpy as np

TRUST_FLOOR = 500
DECAY_IMPRESSIONS_FLOOR = 100
DECAY_STALE_DAYS = 180

visible = df["impressions_90d"] >= TRUST_FLOOR

is_ctr_gap = visible & (df["ctr"] > 0) & (df["ctr"] < df["tier_avg_ctr"])
is_tracking_gap = visible & (df["ctr"] == 0)
is_decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) \
                & (df["content_age_days"] >= 180) \
                & (df["days_since_last_update"] >= DECAY_STALE_DAYS) \
                & (df["impressions_90d"] >= DECAY_IMPRESSIONS_FLOOR)

conditions = [
    is_ctr_gap & is_decay_risk,
    is_ctr_gap,
    is_tracking_gap,
    is_decay_risk,
]
reason_codes = [
    "ctr_below_tier_visible_and_decay_risk",
    "ctr_below_tier_visible",
    "possible_tracking_gap",
    "page_one_decay_risk",
]
actions = [
    "priority_review_ctr_and_refresh",
    "review_for_ctr_gap",
    "verify_tracking_setup",
    "schedule_content_refresh",
]

df["reason_code"] = np.select(conditions, reason_codes, default="none")
df["action"] = np.select(conditions, actions, default="none")
df["ctr_gap_abs"] = (df["ctr"] - df["tier_avg_ctr"]).abs()
df["rank_score"] = df["ctr_gap_abs"] * np.log1p(df["impressions_90d"])
df["estimated_impact"] = df["impressions_90d"] * df["ctr_gap_abs"]

queue = df[df["reason_code"] != "none"].copy()
queue = queue.sort_values(["reason_code", "rank_score"], ascending=[True, False])

print("Archetype counts, with the days_since_last_update fix applied:")
print(queue["reason_code"].value_counts())
print()
print(f"Total flagged: {len(queue):,} of {len(df):,} ({len(queue)/len(df):.1%})")
print()

refresh_rows = queue[queue["reason_code"].isin(["page_one_decay_risk", "ctr_below_tier_visible_and_decay_risk"])]
recently_updated = refresh_rows[refresh_rows["days_since_last_update"] < 30]
print(f"Refresh-flagged rows updated in the last 30 days: {len(recently_updated):,} of {len(refresh_rows):,}")

Archetype counts, with the days_since_last_update fix applied:
reason_code
ctr_below_tier_visible                   8476
possible_tracking_gap                    2538
page_one_decay_risk                        13
ctr_below_tier_visible_and_decay_risk       2
Name: count, dtype: int64

Total flagged: 11,029 of 30,000 (36.8%)

Refresh-flagged rows updated in the last 30 days: 0 of 15


In [17]:
refresh_rows = queue[queue["reason_code"].isin(["page_one_decay_risk", "ctr_below_tier_visible_and_decay_risk"])]
recently_updated = refresh_rows[refresh_rows["days_since_last_update"] < 30]
print(f"Archetype counts:")
print(queue["reason_code"].value_counts())
print()
print(f"Refresh-flagged rows updated in the last 30 days: {len(recently_updated):,} of {len(refresh_rows):,}")

Archetype counts:
reason_code
ctr_below_tier_visible                   5862
ctr_below_tier_visible_and_decay_risk    2616
possible_tracking_gap                    2538
page_one_decay_risk                      2047
Name: count, dtype: int64

Refresh-flagged rows updated in the last 30 days: 2,425 of 4,663


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.